# Student Name: Isaiah Andres
# Student Number: C00286361

## Introduction:
In this notebook, a Recurrent Neural Network (RNN) model will be trained to predict the closing price of Nvidia stock data based on the last 3 years, retrieved from the Yahoo Finance API. The features used to train the model are the opening stock of each day, the closing stock of each day, the highest and lowest a stock has reached of each day and the volume of shares traded of each day. 

# Importing and Preprocessing Data
The cell below defines the ticker for Nvidia and then retrieves it's data from the last three years. The features are also normalised using a min max scaler to scale the data between 0 and 1 as a Long Short Term Memory (LSTM) RNN would work better if the values are bound between 0 and 1 to suit tanh and sigmoid activation functions within the model better whereas another method such as a Z Score normalisation is unbounded and may not be as suitable. Scaling can also prevent higher values from dominating smaller ones. The features are also split into a train test split, specifically with the shuffle turned off to keep the timeframe of the data is in chronological order and data with null values are dropped. 

In [25]:
import yfinance as yf
import pandas as pd 
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential  
from tensorflow.keras.layers import LSTM, Dense, Dropout  
from sklearn.metrics import mean_squared_error, mean_absolute_error
import math

# Define the ticker symbol
ticker_symbol = "NVDA"

ticker = yf.Ticker(ticker_symbol)

# Fetch historical market data for the last 3 years
nvda_historical_data = ticker.history(period="3y")  

# Display a summary of the fetched data
print(f"Summary of Historical Data for {ticker_symbol}:")
print(nvda_historical_data[['Open', 'High', 'Low', 'Close', 'Volume']])

nvda_historical_data = nvda_historical_data.dropna()

features = nvda_historical_data[['Open', 'High', 'Low', 'Close', 'Volume']]

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(features)

scaled_df = pd.DataFrame(scaled_data, 
                         columns=features.columns, 
                         index=features.index)

X_train, X_test = train_test_split(scaled_df, test_size=0.2, shuffle=False)

Summary of Historical Data for NVDA:
                                 Open        High         Low       Close  \
Date                                                                        
2023-03-24 00:00:00-04:00   27.007096   27.142975   26.331693   26.755318   
2023-03-27 00:00:00-04:00   26.813267   26.976123   26.341684   26.507538   
2023-03-28 00:00:00-04:00   26.423608   26.489550   25.827136   26.386641   
2023-03-29 00:00:00-04:00   26.801273   27.054047   26.573474   26.960131   
2023-03-30 00:00:00-04:00   27.204919   27.474680   27.078030   27.358782   
...                               ...         ...         ...         ...   
2026-03-18 00:00:00-04:00  182.479996  183.380005  180.330002  180.399994   
2026-03-19 00:00:00-04:00  178.009995  179.979996  175.789993  178.559998   
2026-03-20 00:00:00-04:00  178.000000  178.259995  171.720001  172.699997   
2026-03-23 00:00:00-04:00  177.259995  178.369995  174.759995  175.639999   
2026-03-24 00:00:00-04:00  174.761993  

# Training The Model
Once the data has been preprocessed, the data is further prepared by creating sliding windows of sequential features over the last three years of data, e.g the 5 features and the prediction of the closing stock price, into overlapping sixty day periods alongside the closing price of the stock the day after the 30th day of each split in order to train the model to learn past price movements. This is done by splitting the data into multiple training samples instead of one large input consisting of the full three years of data with one output. The model itself has two LSTM layers with dropout applied after each layer used to prevent overfitting with a dense layer and is trained over 100 epochs. 

In [26]:
#function splits the X and Y values into 30 day intervals representing smaller datasets that the model can use to predict the result of the next day
#The result predicted is the closing pricing of microsoft stocks
def create_dataset(dataset, look_back=30):
    X, Y = [], []
    for i in range(len(dataset) - look_back - 1):
        X.append(dataset.iloc[i:i + look_back].values)
        Y.append(dataset['Close'].iloc[i + look_back])
    return np.array(X), np.array(Y)

look_back = 30
X_train, Y_train = create_dataset(X_train, look_back)
X_test,  Y_test  = create_dataset(X_test,  look_back)

model = Sequential([
    LSTM(32, return_sequences=True, input_shape=(look_back, 5)),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(1)
])
model.compile(loss='mean_squared_error', optimizer='adam')

model.fit(X_train, Y_train, epochs=100, batch_size=64, verbose=1)

Epoch 1/100


C:\Users\nasha\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - loss: 0.0735
Epoch 2/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0165
Epoch 3/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 0.0096
Epoch 4/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0087
Epoch 5/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0067
Epoch 6/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0064
Epoch 7/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0056
Epoch 8/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0052
Epoch 9/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0055
Epoch 10/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0051
Epoch 11/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0050
Epoch 12/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0050
Epoch 13/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0045
Epoch 14/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0048
Epoch 15/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0043
Epoch 16/100
9/9 ━━━━━━━━━━━━━━

# Evaluating The Results
The function invert_close, takes the five features from the scaler and then flattens the fourth column and inverses the min max scaling performed in order to retrieve the actual value of the prices. Previously, the model achieved a training Root Mean Squared Error (RMSE) of 5.522 USD and a testing RMSE of 11.947 USD, along with a training Mean Absolute Error (MAE) of 4.143 USD and a testing MAE of 10.767 USD. The error on the test set is almost double than that of the training set showing that the model is more than likely overfitting and not performing well with unseen data as the RMSE measures the average error between the predicted and actual values and the MAE shows the average size of mistakes in a collection of predictions.

Upon tuning the parameters of the model such as lowering the number of neurons used to try and prevent overfitting, decreasing the number of days from 60 to 30 when constructing a dataset out of the time series data and most of all, increasing the size of the training set from a 70% split to an 80% split, the testing RMSE and MAE are now 6.223 USD and 4.865 USD respectively as well as a smaller gap between the training and testing RMSE and MAE as it reduced from around 6 USD to less than 1 USD, showing better performance on unseen data when predicting the closing price of a given stock such as Nvidia.  

In [27]:
# Inversing all ['Close'] values from their scaled values to read easier
# Takes 5D input since 5 features were scaled on the scaler previously but ['Close'] was the only one of interest as the result
def invert_close(scaler, values):
    dummy = np.zeros((len(values), 5))
    dummy[:, 3] = values.flatten()
    return scaler.inverse_transform(dummy)[:, 3] #Inversing all ['Close'] 
    
train_predict  = invert_close(scaler, model.predict(X_train).flatten())
Y_train_actual = invert_close(scaler, Y_train)

test_predict   = invert_close(scaler, model.predict(X_test).flatten())
Y_test_actual  = invert_close(scaler, Y_test)

# Calculate root mean squared error (RMSE)
train_rmse = math.sqrt(mean_squared_error(Y_train_actual, train_predict))
test_rmse  = math.sqrt(mean_squared_error(Y_test_actual, test_predict))

print('Train RMSE: %.3f' % (train_rmse))
print('Test RMSE: %.3f' % (test_rmse))

# Calculate mean absolute error (MAE)
train_mae  = mean_absolute_error(Y_train_actual, train_predict)
test_mae   = mean_absolute_error(Y_test_actual, test_predict)
print('Train MAE: %.3f' % (train_mae))
print('Test MAE: %.3f' % (test_mae))

18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
Train RMSE: 5.630
Test RMSE: 6.223
Train MAE: 4.097
Test MAE: 4.865


# Conclusion
In conclusion, the model performs decently well when predicting the closing price of a stock after tuning the model, as without tuning, the model seems to overfit as seen with how poorly it predicted the closing stock price when using unseen data. Following the tuning of the model's parameters the 6 USD gap between the training and testing RMSE and MAE has been reduced to less than a 1 USD gap on unseen data in the case of Nvidia. The final results of the model's performance returned a training RMSE of 5.630 USD and testing RMSE of 6.223 USD, and training MAE of 4.097 USD and testing MAE of 4.865 USD in comparison to the previous testing RMSE and MAE of 11.947 USD and 10.767 USD respectively.

The data however is retrieved from the last three years and doesn't take into account any external factors about the stock data in general so the model would not be the most reliable predictor for deciding on whether to buy or sell stock. 